# PEDE / Passos Mágicos — Analytics 2022–2024
## Objetivo
Analisar IAN, IDA, IEG, IAA, IPS, IPP, IPV e INDE e construir um modelo preditivo de risco de defasagem.

### Premissas
- O dicionário define IAN como adequação ao nível atual e Defasagem como o nível de defasagem. 
- O INDE é uma ponderação de IAN, IDA, IEG, IAA, IPS, IPP e IPV.
- A base é anual; portanto, “ao longo do ano” é tratado como evolução entre ciclos anuais.
- O dicionário não define severidade da Defasagem. Para fins analíticos: -1 = leve, -2 = moderada, <= -3 = severa.
- Correlação/regressão mostram associação, não causalidade.


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt

# caso o arquivo seja executado no COLAB:
# BASE = "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# caso o arquivo seja executado localmente
BASE = "C:/Users/eduar/Desktop/ADS/Dados/Pós Tech/5.Final/Streamlit/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
raw = {y: pd.read_excel(BASE, sheet_name=f"PEDE{y}") for y in ["2022","2023","2024"]}

def harmonize(df, year):
    y=int(year); d=df.copy()
    d["defas"]=d["Defas"] if y==2022 else d["Defasagem"]
    d["inde"]=d["INDE 22"] if y==2022 else d[f"INDE {y}"]
    d["pedra"]=d["Pedra 22"] if y==2022 else d[f"Pedra {y}"]
    d["IPP"]=d["IPP"] if "IPP" in d.columns else np.nan
    d["idade_num"]=pd.to_numeric(d["Idade 22"] if y==2022 else d["Idade"],errors="coerce")
    d["pedra_cat"]=d["pedra"].astype(str).replace({"Agata":"Ágata"})
    return d

years={int(y):harmonize(raw[y],y) for y in raw}


## 1. IAN e Defasagem

In [3]:
for y,g in years.items():
    print(y,
          "IAN:",round(g["IAN"].mean(),3),
          "defasagem <0:",round((g["defas"]<0).mean(),3),
          "moderada+ <=-2:",round((g["defas"]<=-2).mean(),3),
          "severa <=-3:",round((g["defas"]<=-3).mean(),3))


2022 IAN: 6.424 defasagem <0: 0.699 moderada+ <=-2: 0.222 severa <=-3: 0.033
2023 IAN: 7.244 defasagem <0: 0.544 moderada+ <=-2: 0.142 severa <=-3: 0.014
2024 IAN: 7.684 defasagem <0: 0.462 moderada+ <=-2: 0.08 severa <=-3: 0.003


## 2. IDA, IEG, IPV e INDE

In [ ]:
for y,g in years.items():
    print(y,
          "IDA:",round(g["IDA"].mean(),3),
          "IEG:",round(g["IEG"].mean(),3),
          "IPV:",round(g["IPV"].mean(),3),
          "INDE:",round(pd.to_numeric(g["inde"],errors="coerce").mean(),3))


## 3. IEG x IDA/IPV

In [ ]:
for y,g in years.items():
    for x,z in [("IEG","IDA"),("IEG","IPV")]:
        q=g[[x,z]].dropna()
        print(y,x,z,
              "Pearson:",round(pearsonr(q[x],q[z]).statistic,3),
              "Spearman:",round(spearmanr(q[x],q[z]).statistic,3))


## 4. IAA x IDA

In [ ]:
for y,g in years.items():
    gap=g["IAA"]-g["IDA"]
    print(y,
          "gap médio:",round(gap.mean(),3),
          "IAA >= IDA+2:",round((gap>=2).mean(),3),
          "IDA >= IAA+2:",round((gap<=-2).mean(),3))


## 5. IPS como antecedente

In [ ]:
idx={y:years[y].set_index("RA") for y in years}
for y1,y2 in [(2022,2023),(2023,2024)]:
    common=idx[y1].index.intersection(idx[y2].index)
    a,b=idx[y1].loc[common],idx[y2].loc[common]
    for x in ["IPS","IEG","IAA","IAN"]:
        q=pd.DataFrame({"dx":b[x]-a[x],"dIDA":b["IDA"]-a["IDA"],"dIEG":b["IEG"]-a["IEG"]}).dropna()
        print(y1,"->",y2,x,
              "corr ΔIDA:",round(pearsonr(q.dx,q.dIDA).statistic,3),
              "corr ΔIEG:",round(pearsonr(q.dx,q.dIEG).statistic,3))


## 6. IPP x IAN

In [ ]:
for y in [2023,2024]:
    q=years[y].dropna(subset=["IAN","IPP"]).groupby("IAN").agg(
        alunos=("RA","size"),IPP=("IPP","mean"),IDA=("IDA","mean"),
        defasagem=("defas",lambda x:(x<0).mean()))
    display(q.round(3))


## 7. IPV — regressão multivariada

In [ ]:
for y,g in years.items():
    cols=[c for c in ["IEG","IDA","IAA","IPS","IPP","IAN"] if g[c].notna().sum()>100]
    q=g[cols+["IPV"]].dropna()
    X=StandardScaler().fit_transform(q[cols])
    lr=LinearRegression().fit(X,q["IPV"])
    print(y,"R2:",round(lr.score(X,q["IPV"]),3))
    display(pd.Series(lr.coef_,index=cols).sort_values(key=np.abs,ascending=False).round(3))


## 8. Multidimensionalidade

In [ ]:
for y in [2023,2024]:
    g=years[y].copy(); cols=["IDA","IEG","IPS","IPP"]; med=g[cols].median()
    q=g[cols+["inde"]].dropna()
    q["n_acima_mediana"]=(q[cols]>=med).sum(axis=1)
    display(q.groupby("n_acima_mediana")["inde"].agg(["count","mean"]).round(3))


## 9. Efetividade — cohort longitudinal

In [ ]:
common=idx[2022].index.intersection(idx[2023].index).intersection(idx[2024].index)
print("alunos nos 3 ciclos:",len(common))
for y in [2022,2023,2024]:
    g=idx[y].loc[common]
    print(y,
          "IAN",round(g.IAN.mean(),3),
          "IDA",round(g.IDA.mean(),3),
          "IEG",round(g.IEG.mean(),3),
          "IPV",round(g.IPV.mean(),3),
          "INDE",round(pd.to_numeric(g.inde,errors="coerce").mean(),3),
          "defasagem",round((g.defas<0).mean(),3))


## 10. Machine Learning — risco de defasagem
### Target
`1` se a Defasagem do ano seguinte for `< 0`; `0` caso contrário.

### Estratégia temporal
- Treino: 2022 → 2023
- Teste: 2023 → 2024

### Features
IAN, IDA, IEG, IAA, IPS, IPV e Pedra. O INDE fica fora para evitar dependência direta de um índice que já agrega os indicadores.


In [ ]:
def transition(y):
    cur=years[y].set_index("RA"); nxt=years[y+1].set_index("RA")
    common=cur.index.intersection(nxt.index)
    return cur.loc[common].copy(),(nxt.loc[common,"defas"]<0).astype(int)

X_train,y_train=transition(2022)
X_test,y_test=transition(2023)

features=["IAN","IDA","IEG","IAA","IPS","IPV","pedra_cat"]
num=[c for c in features if c!="pedra_cat"]

pre=ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),num),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("ohe",OneHotEncoder(handle_unknown="ignore"))]),["pedra_cat"])
])

model=Pipeline([
    ("preprocess",pre),
    ("model",LogisticRegression(max_iter=2000,class_weight="balanced"))
])

model.fit(X_train[features],y_train)
prob=model.predict_proba(X_test[features])[:,1]
pred30=(prob>=.30).astype(int)
pred50=(prob>=.50).astype(int)


### Avaliação

In [ ]:
def metricas(y,prob,pred):
    return pd.Series({
        "AUC":roc_auc_score(y,prob),
        "Accuracy":accuracy_score(y,pred),
        "Precision":precision_score(y,pred),
        "Recall":recall_score(y,pred),
        "F1":f1_score(y,pred)
    })

display(metricas(y_test,prob,pred30).round(3))
display(metricas(y_test,prob,pred50).round(3))
display(pd.DataFrame(confusion_matrix(y_test,pred30),
                     index=["Real sem risco","Real em risco"],
                     columns=["Previsto sem risco","Previsto em risco"]))


### Faixas de risco
- Baixo: <30%
- Médio: 30–70%
- Alto: >70%

Na validação, a taxa real de risco foi aproximadamente 26%, 56% e 85% nessas faixas. As faixas servem para priorização de acompanhamento, não para diagnóstico.


In [ ]:
bands=pd.cut(prob,[0,.3,.7,1],labels=["Baixo (<30%)","Médio (30–70%)","Alto (>70%)"],include_lowest=True)
display(pd.DataFrame({"Faixa":bands,"Risco_real":y_test.values}).groupby("Faixa",observed=False).agg(
    alunos=("Risco_real","size"),risco_real=("Risco_real","mean")).round(3))


## 11. Recomendações
1. Criar régua de alerta por aluno com probabilidade de risco.
2. Monitorar ΔIAN, ΔIDA, ΔIEG e ΔIPV entre ciclos.
3. Usar IAA–IDA como sinal de metacognição/feedback.
4. Integrar IPS e IPP aos sinais acadêmicos, evitando uso isolado.
5. Usar cohort para separar efeito de composição da base.
6. Evoluir o modelo com histórico de 3+ ciclos e variáveis textuais codificadas.
